In [1]:
!pip -q install pyspark

In [2]:
from google.colab import files
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("Lazy Execution").getOrCreate()

In [4]:
uploaded = files.upload()


Saving lazy_execution_sales_25000.csv to lazy_execution_sales_25000.csv


In [5]:
uploaded = files.upload()

Saving lazy_execution_products.csv to lazy_execution_products.csv


In [6]:
uploaded = files.upload()

Saving lazy_execution_stores.csv to lazy_execution_stores.csv


In [7]:
sales_df = spark.read.csv("lazy_execution_sales_25000.csv", header=True, inferSchema=True)
products_df = spark.read.csv("lazy_execution_products.csv", header=True, inferSchema=True)
stores_df = spark.read.csv("lazy_execution_stores.csv", header=True, inferSchema=True)

In [8]:
high_qty_df = sales_df.filter(F.col("quantity") > 5)

In [9]:
sales_df.select("order_id", "product_id", "quantity", "unit_price").show()

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
| O000001|      P002|       3|      2999|
| O000002|      P014|       6|     29999|
| O000003|      P008|       6|     22999|
| O000004|      P008|       7|     22999|
| O000005|      P007|       2|     79999|
| O000006|      P005|       8|      2499|
| O000007|      P010|       8|      3999|
| O000008|      P002|       9|      2999|
| O000009|      P007|       5|     79999|
| O000010|      P006|       5|     55999|
| O000011|      P007|       1|     79999|
| O000012|      P010|       9|      3999|
| O000013|      P013|       6|      5999|
| O000014|      P005|       8|      2499|
| O000015|      P007|       7|     79999|
| O000016|      P006|       1|     55999|
| O000017|      P004|       9|     16999|
| O000018|      P012|       7|      6999|
| O000019|      P014|       5|     29999|
| O000020|      P010|       7|      3999|
+--------+----------+--------+----

In [10]:
sales_df.filter(F.col("unit_price") > 20000).count()

6560

In [11]:
gross_df = sales_df.withColumn("gross_amount", F.col("quantity") * F.col("unit_price"))

gross_df.explain()

gross_df.select("order_id", "quantity", "unit_price", "gross_amount").show()

== Physical Plan ==
*(1) Project [order_id#17, order_date#18, store_id#19, product_id#20, quantity#21, unit_price#22, discount_pct#23, payment_method#24, order_status#25, sales_channel#26, (quantity#21 * unit_price#22) AS gross_amount#103]
+- FileScan csv [order_id#17,order_date#18,store_id#19,product_id#20,quantity#21,unit_price#22,discount_pct#23,payment_method#24,order_status#25,sales_channel#26] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...


+--------+--------+----------+------------+
|order_id|quantity|unit_price|gross_amount|
+--------+--------+----------+------------+
| O000001|       3|      2999|        8997|
| O000002|       6|     29999|      179994|
| O000003|       6|     22999|      137994|
| O000004|       7|     22999|      160993|
| O

In [12]:
top_completed = sales_df.filter(F.col("order_status") == "Completed") \
    .orderBy(F.col("unit_price").desc()) \
    .take(10)

for row in top_completed:
    print(row)

Row(order_id='O000085', order_date=datetime.date(2026, 4, 22), store_id='S03', product_id='P007', quantity=4, unit_price=79999, discount_pct=0, payment_method='Card', order_status='Completed', sales_channel='Online')
Row(order_id='O000290', order_date=datetime.date(2026, 6, 26), store_id='S06', product_id='P007', quantity=3, unit_price=79999, discount_pct=10, payment_method='Net Banking', order_status='Completed', sales_channel='Online')
Row(order_id='O000098', order_date=datetime.date(2026, 4, 15), store_id='S05', product_id='P007', quantity=1, unit_price=79999, discount_pct=5, payment_method='Net Banking', order_status='Completed', sales_channel='Online')
Row(order_id='O000005', order_date=datetime.date(2026, 3, 14), store_id='S06', product_id='P007', quantity=2, unit_price=79999, discount_pct=0, payment_method='Cash', order_status='Completed', sales_channel='Online')
Row(order_id='O000107', order_date=datetime.date(2026, 5, 24), store_id='S05', product_id='P007', quantity=9, unit_pr

In [13]:
sales_df.filter(F.col("order_status") == "Completed") \
    .withColumn("gross_amount", F.col("quantity") * F.col("unit_price")) \
    .select("order_id", "product_id", "quantity", "unit_price", "gross_amount") \
    .orderBy(F.col("gross_amount").desc()) \
    .show()

+--------+----------+--------+----------+------------+
|order_id|product_id|quantity|unit_price|gross_amount|
+--------+----------+--------+----------+------------+
| O002603|      P007|       9|     79999|      719991|
| O006170|      P007|       9|     79999|      719991|
| O002693|      P007|       9|     79999|      719991|
| O000107|      P007|       9|     79999|      719991|
| O002841|      P007|       9|     79999|      719991|
| O000367|      P007|       9|     79999|      719991|
| O002940|      P007|       9|     79999|      719991|
| O000527|      P007|       9|     79999|      719991|
| O003108|      P007|       9|     79999|      719991|
| O000690|      P007|       9|     79999|      719991|
| O003542|      P007|       9|     79999|      719991|
| O001201|      P007|       9|     79999|      719991|
| O003884|      P007|       9|     79999|      719991|
| O001300|      P007|       9|     79999|      719991|
| O003897|      P007|       9|     79999|      719991|
| O001796|

In [14]:
sales_df.filter(F.col("discount_pct") >= 10) \
    .withColumn(
        "discounted_amount",
        F.round(F.col("quantity") * F.col("unit_price") * (1 - F.col("discount_pct") / 100), 2)
    ) \
    .select("order_id", "quantity", "unit_price", "discount_pct", "discounted_amount") \
    .orderBy(F.col("discounted_amount").desc()) \
    .show(15)

+--------+--------+----------+------------+-----------------+
|order_id|quantity|unit_price|discount_pct|discounted_amount|
+--------+--------+----------+------------+-----------------+
| O006653|       9|     79999|          10|         647991.9|
| O011893|       9|     79999|          10|         647991.9|
| O006699|       9|     79999|          10|         647991.9|
| O003108|       9|     79999|          10|         647991.9|
| O007743|       9|     79999|          10|         647991.9|
| O000264|       9|     79999|          10|         647991.9|
| O009316|       9|     79999|          10|         647991.9|
| O000367|       9|     79999|          10|         647991.9|
| O009625|       9|     79999|          10|         647991.9|
| O003974|       9|     79999|          10|         647991.9|
| O009646|       9|     79999|          10|         647991.9|
| O002907|       9|     79999|          10|         647991.9|
| O010062|       9|     79999|          10|         647991.9|
| O00494

In [15]:
sales_df.groupBy("payment_method") \
    .agg(F.sum("quantity").alias("total_quantity")) \
    .orderBy(F.col("total_quantity").desc()) \
    .show()

+--------------+--------------+
|payment_method|total_quantity|
+--------------+--------------+
|          Cash|         31754|
|          Card|         31327|
|           UPI|         30894|
|   Net Banking|         30889|
+--------------+--------------+



In [16]:
sales_df.groupBy("sales_channel", "order_status") \
    .agg(F.count("*").alias("transaction_count")) \
    .orderBy("sales_channel", "order_status") \
    .show()

+-------------+------------+-----------------+
|sales_channel|order_status|transaction_count|
+-------------+------------+-----------------+
|     In Store|   Cancelled|              592|
|     In Store|   Completed|            10978|
|     In Store|    Returned|             1013|
|       Online|   Cancelled|              633|
|       Online|   Completed|            10797|
|       Online|    Returned|              987|
+-------------+------------+-----------------+



In [17]:
one_record = sales_df.select("order_id", "store_id", "product_id", "quantity") \
    .drop("store_id") \
    .first()

print(one_record)

Row(order_id='O000001', product_id='P002', quantity=3)


In [18]:
sales_df.join(products_df, on="product_id", how="inner") \
    .select("product_name", "category", "quantity", "unit_price") \
    .show(20)

+-------------------+-----------+--------+----------+
|       product_name|   category|quantity|unit_price|
+-------------------+-----------+--------+----------+
|Mechanical Keyboard|Accessories|       3|      2999|
|      Smartphone 5G|    Mobiles|       6|     29999|
|     Android Tablet|    Tablets|       6|     22999|
|     Android Tablet|    Tablets|       7|     22999|
|          Laptop 16|  Computers|       2|     79999|
|          Webcam HD|Accessories|       8|      2499|
|  Bluetooth Speaker|      Audio|       8|      3999|
|Mechanical Keyboard|Accessories|       9|      2999|
|          Laptop 16|  Computers|       5|     79999|
|          Laptop 14|  Computers|       5|     55999|
|          Laptop 16|  Computers|       1|     79999|
|  Bluetooth Speaker|      Audio|       9|      3999|
|   External HDD 2TB|    Storage|       6|      5999|
|          Webcam HD|Accessories|       8|      2499|
|          Laptop 16|  Computers|       7|     79999|
|          Laptop 14|  Compu

In [19]:
sales_df.join(stores_df, on="store_id", how="inner") \
    .groupBy("city") \
    .agg(F.sum("quantity").alias("total_quantity")) \
    .orderBy(F.col("total_quantity").desc()) \
    .show()

+---------+--------------+
|     city|total_quantity|
+---------+--------------+
|   Jaipur|         16019|
|Hyderabad|         15784|
|  Chennai|         15727|
|Bengaluru|         15718|
|  Kolkata|         15619|
|   Mumbai|         15489|
|     Pune|         15407|
|    Delhi|         15101|
+---------+--------------+



In [20]:
sales_df.join(products_df, on="product_id", how="inner") \
    .join(stores_df, on="store_id", how="inner") \
    .filter(F.col("order_status") == "Completed") \
    .withColumn("revenue", F.col("quantity") * F.col("unit_price")) \
    .groupBy("category", "region") \
    .agg(F.sum("revenue").alias("total_revenue")) \
    .orderBy("category", "region") \
    .show(

    )

+-----------+------+-------------+
|   category|region|total_revenue|
+-----------+------+-------------+
|Accessories|  East|      9093361|
|Accessories| North|     18463455|
|Accessories| South|     28139018|
|Accessories|  West|     18351560|
|      Audio|  East|     12043135|
|      Audio| North|     24327211|
|      Audio| South|     34629665|
|      Audio|  West|     25317111|
|  Computers|  East|    122990192|
|  Computers| North|    240756425|
|  Computers| South|    371954527|
|  Computers|  West|    221092729|
|   Displays|  East|     13616199|
|   Displays| North|     32468090|
|   Displays| South|     45761308|
|   Displays|  West|     28286336|
|    Mobiles|  East|     41178126|
|    Mobiles| North|     75482543|
|    Mobiles| South|    120674524|
|    Mobiles|  West|     80372388|
+-----------+------+-------------+
only showing top 20 rows


In [22]:
sales_df.filter(F.col("order_status") == "Completed") \
    .join(products_df, on="product_id", how="inner") \
    .withColumn(
        "discounted_revenue",
        F.col("quantity") * F.col("unit_price") * (1 - F.col("discount_pct") / 100)
    ) \
    .groupBy("product_name") \
    .agg(F.round(F.sum("discounted_revenue"), 2).alias("total_revenue")) \
    .orderBy(F.col("total_revenue").desc()) \
    .show(5)

+---------------+--------------+
|   product_name| total_revenue|
+---------------+--------------+
|      Laptop 16|5.2272546585E8|
|      Laptop 14| 3.830051605E8|
|  Smartphone 5G|2.0604063175E8|
| Android Tablet| 1.504065603E8|
|27-inch Monitor|1.1406074015E8|
+---------------+--------------+
only showing top 5 rows


In [23]:
step1 = sales_df.filter(F.col("order_status") == "Completed")
print("filter defined")

step2 = step1.select("product_id", "quantity", "unit_price")
print("select defined")

step3 = step2.withColumn("gross_amount", F.col("quantity") * F.col("unit_price"))
print("withColumn defined")

step4 = step3.groupBy("product_id").agg(F.sum("gross_amount").alias("total_gross"))
print("groupBy defined")

step5 = step4.orderBy(F.col("total_gross").desc())
print("orderBy defined")

filter defined
select defined
withColumn defined
groupBy defined
orderBy defined


In [24]:
step5.show(5)

+----------+-----------+
|product_id|total_gross|
+----------+-----------+
|      P007|  552313096|
|      P006|  404480777|
|      P014|  217222759|
|      P008|  158555106|
|      P004|  120131933|
+----------+-----------+
only showing top 5 rows


In [25]:
filtered_df = sales_df.filter(F.col("unit_price") > 20000)

filtered_df.explain()

== Physical Plan ==
*(1) Filter (isnotnull(unit_price#22) AND (unit_price#22 > 20000))
+- FileScan csv [order_id#17,order_date#18,store_id#19,product_id#20,quantity#21,unit_price#22,discount_pct#23,payment_method#24,order_status#25,sales_channel#26] Batched: false, DataFilters: [isnotnull(unit_price#22), (unit_price#22 > 20000)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [IsNotNull(unit_price), GreaterThan(unit_price,20000)], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...




In [26]:
filtered_df.show()

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Completed|       Online|
| O000009|2026-01-15|     S03|      P007|       5|     79999|          15|          Cash|   Completed|     In Store|
| O000010|2026-06-01|     S02|      P006|       5|     55999|   

In [27]:
filtered_df.count()

6560

In [28]:
category_totals = sales_df.join(products_df, on="product_id", how="inner") \
    .groupBy("category") \
    .agg(F.sum("quantity").alias("total_quantity")) \
    .orderBy(F.col("total_quantity").desc()) \
    .collect()

for row in category_totals:
    print(row["category"], row["total_quantity"])

Accessories 41839
Storage 17251
Audio 17159
Mobiles 16572
Computers 16258
Tablets 7915
Displays 7870


In [29]:
sales_df.filter(F.col("order_status") == "Completed") \
    .withColumn(
        "discounted_revenue",
        F.round(F.col("quantity") * F.col("unit_price") * (1 - F.col("discount_pct") / 100), 2)
    ) \
    .write.mode("overwrite").parquet("completed_discounted_revenue.parquet")

In [30]:
spark.read.parquet("completed_discounted_revenue.parquet").count()

21775